# Deep-Guard — Large-Scale AI Media Detection Training

Trains one model to answer a single question about **any** uploaded
image or video frame: *was this made by AI, or captured by a camera?*
Not faces-only. Not one generator. Any subject.

## What this trains on

Two datasets are combined, giving roughly **200,000 images** spanning
five different generator families:

| Source | Images | What it contributes |
|---|---|---|
| `tristanzhang32/ai-generated-images-vs-real-images` | 60,000 | AI from **Stable Diffusion, MidJourney, DALL-E**. Real from Pexels, Unsplash, WikiArt. General subjects: scenery, animals, objects, artwork |
| `xhlulu/140k-real-and-fake-faces` | 140,000 | AI faces from **StyleGAN** (a GAN, not a diffusion model). Real faces from Flickr/FFHQ |

**Why combine rather than pick the biggest one?** Published research on
this exact problem is consistent: detectors trained on a single
generator learn *that generator's* quirks instead of a general sense of
"AI-ness," and collapse to near-random accuracy on generators they have
never seen. Generator *diversity* matters more than raw image count.
Combining diffusion models with a GAN, and faces with general scenes, is
what gives the model a chance to generalise to generators that did not
exist when it was trained.

## Why this also covers AI-generated video

Modern AI video tools (Sora, Runway, Kling, Veo) are diffusion models —
the same underlying technology as Stable Diffusion and MidJourney, just
extended over time. Individual frames pulled from their output carry the
same statistical fingerprints as diffusion-generated still images. So a
strong general image detector transfers to AI-generated video frames,
which is exactly how your app already analyses video: sample frames,
score each one, aggregate.

**Honest limitation to state in your report:** face-*swap* deepfakes
(a real video with someone else's face pasted on) are a different
problem. They leave blending seams at the face boundary rather than
whole-image generation artifacts. This model will be weaker on those.
Covering them properly needs a face-swap video dataset such as
FaceForensics++, which is a sensible next milestone.

## Also fixed here

The original notebook's fine-tuning stage only unfroze ~10% of the
network, at a learning rate so low it barely moved — which is why
accuracy was stuck at 85.67%. This notebook fine-tunes the whole
network properly.

---

### Before you run

1. **Runtime → Change runtime type → T4 GPU → Save.**
2. Expect **2-4 hours** total. Colab disconnects after ~90 minutes of
   *browser* inactivity, so leave the tab open and check in periodically.
3. Step 3 offers to mount your Google Drive. **Say yes.** Checkpoints get
   copied there after every improvement, so a disconnect costs you time,
   not work.

## Step 1 — Check the GPU and available disk

In [ ]:
!nvidia-smi
print()
!df -h /content | tail -1
print()
print("You need roughly 60 GB free under /content for both datasets.")
print("If free space is under 60 GB, set USE_LARGE_DATASET_ONLY = True in Step 4.")

## Step 2 — Install the Kaggle tool

In [ ]:
!pip install -q kaggle==1.6.17

## Step 3 — Credentials and Drive backup

Two things here: your Kaggle key (so the datasets can download), and
optionally your Google Drive (so long training runs survive a
disconnect). Mounting Drive opens a permission popup — approve it.

In [ ]:
from google.colab import files

print("Select your kaggle.json file below.")
uploaded = files.upload()

assert "kaggle.json" in uploaded, (
    "No file named kaggle.json was uploaded. Re-run this cell and pick the "
    "correct file. It must be named exactly kaggle.json."
)

!mkdir -p ~/.kaggle
!mv kaggle.json ~/.kaggle/kaggle.json
!chmod 600 ~/.kaggle/kaggle.json
print("Kaggle credentials installed.")

In [ ]:
# Strongly recommended for a multi-hour run.
USE_DRIVE_BACKUP = True

DRIVE_BACKUP_PATH = None
if USE_DRIVE_BACKUP:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        DRIVE_BACKUP_PATH = "/content/drive/MyDrive/deepguard_bouncer.pth"
        print(f"\nCheckpoints will be backed up to: {DRIVE_BACKUP_PATH}")
    except Exception as e:
        print(f"Drive mount failed ({e}). Continuing without backup.")
        DRIVE_BACKUP_PATH = None
else:
    print("Drive backup disabled. A disconnect will lose your progress.")

## Step 4 - Configuration

Switches you may want to change:

- `USE_LARGE_DATASET_ONLY` - set to `True` only as a last resort (e.g. a
  Colab tier with very little disk). This skips the general dataset
  entirely, which means training never sees a single Stable Diffusion /
  MidJourney / DALL-E example -- a previous run proved this is a much
  worse trade than it sounds: the model didn't just miss "some" AI images,
  it lost the ability to flag ANY AI-generated image at all, because
  StyleGAN (the only fake source left) has such a narrow, distinctive look
  that "doesn't look like StyleGAN" became a shortcut for "must be real."
  Leave this `False` unless you have no other option.
- `GENERAL_SAMPLE_PER_CLASS` - instead of downloading and extracting the
  full ~52 GB general dataset (which doesn't fit in this notebook's disk
  budget), this many real and this many fake images are sampled directly
  out of its zip file, chosen randomly so all three generators (Stable
  Diffusion/MidJourney/DALL-E) are represented. The zip itself still has
  to be downloaded in full first (zips can't be read in pieces), but only
  this bounded sample gets extracted, so disk usage stays safe. If your
  session's disk is tight, the actual extracted count may land below this
  target -- the extraction stops cleanly rather than crashing (see
  `MIN_FREE_GB_SAFETY`).
- `MIN_FREE_GB_SAFETY` - the general-dataset extraction stops the moment
  free disk drops below this, so later steps (the other two datasets,
  installing torch/torchvision) always have room. Do not lower this.
- `FACES_CAP_PER_CLASS` - the faces dataset alone has 70,000 images per
  class, which would otherwise vastly outnumber the general dataset's
  small disk-safe sample and drown it out again, the same failure mode as
  before just shifted to a different pair of sources. Capping it keeps the
  two sources from being wildly mismatched in size.
- `GENERAL_FAKE_OVERSAMPLE` / `GENERAL_REAL_OVERSAMPLE` - the general
  sample is small by necessity (disk), so each of its images is repeated
  this many times in the training pool to give it real influence instead
  of being statistically drowned out by the much larger faces dataset.
  Repeats still get independently randomized augmentation each time
  they're sampled (see Step 7), so this isn't the same as training on
  literal duplicate images.
- `INCLUDE_CASUAL_PHOTOS` - adds ~78k casual selfie and everyday-snapshot
  photos to the REAL class, so the model sees ordinary imperfect photos,
  not just studio-quality portraits and stock photography.
- `MAX_PER_CLASS` - caps how many images are used per class after all the
  above. `None` uses everything available.

In [ ]:
USE_LARGE_DATASET_ONLY = False   # True = skip the general dataset entirely (last resort only -- see Step 4)
INCLUDE_CASUAL_PHOTOS = True     # True = also add ~78k casual/selfie real photos

# General-dataset disk-safe sampling (see Step 4 for the full reasoning).
GENERAL_SAMPLE_PER_CLASS = 4500   # target real/fake images to extract from the general dataset's zip
MIN_FREE_GB_SAFETY = 14           # extraction stops cleanly below this, protecting later steps
FACES_CAP_PER_CLASS = 25000       # cap so the 70k-image faces dataset can't drown out the general sample
GENERAL_FAKE_OVERSAMPLE = 4       # repeat each general fake image this many times in the training pool
GENERAL_REAL_OVERSAMPLE = 2       # same idea, real side (less critical, still helps)

MAX_PER_CLASS = None             # e.g. 40000 to cap further, None = use everything from the steps above

EPOCHS_HEAD = 2                  # warm-up epochs (frozen backbone)
EPOCHS_FINETUNE = 6              # full fine-tuning epochs
BATCH_SIZE = 96

print(f"General dataset:      {'SKIPPED' if USE_LARGE_DATASET_ONLY else f'disk-safe sample, target {GENERAL_SAMPLE_PER_CLASS:,}/class'}")
print(f"Casual/selfie photos: {'INCLUDED' if INCLUDE_CASUAL_PHOTOS else 'SKIPPED'}")
print(f"Faces dataset capped at: {FACES_CAP_PER_CLASS:,}/class")
print(f"Images per class:     {'all available' if MAX_PER_CLASS is None else MAX_PER_CLASS}")
print(f"Epochs:               {EPOCHS_HEAD} warm-up + up to {EPOCHS_FINETUNE} fine-tune")

## Step 5 — Download the datasets

Each zip is deleted immediately after extraction, so peak disk usage
stays as low as possible. The large one is ~52 GB and can take 15-30
minutes; the faces one is ~4 GB.

In [ ]:
import os, random, shutil, subprocess, zipfile
from pathlib import Path

DATA_DIR = Path("/content/data")
DATA_DIR.mkdir(exist_ok=True)

def free_gb():
    return shutil.disk_usage("/content").free / 1e9

def fetch(slug, folder_name):
    target = DATA_DIR / folder_name
    if target.exists() and any(target.iterdir()):
        print(f"[skip] {slug} already present at {target}")
        return
    target.mkdir(parents=True, exist_ok=True)
    print(f"\n[download] {slug}  (free disk: {free_gb():.1f} GB)")
    subprocess.run(
        ["kaggle", "datasets", "download", "-d", slug, "-p", str(target)],
        check=True,
    )
    print(f"[extract]  {slug}")
    for zip_path in target.glob("*.zip"):
        subprocess.run(["unzip", "-q", "-o", str(zip_path), "-d", str(target)], check=True)
        zip_path.unlink()          # free the space immediately
    print(f"[done]     {slug}  (free disk: {free_gb():.1f} GB)")


def fetch_partial_by_class(slug, folder_name, per_class_target, min_free_gb):
    """
    Downloads slug's zip in full (unavoidable -- a zip's central directory
    needs the whole file on disk before individual members can be read),
    then extracts only up to `per_class_target` real and `per_class_target`
    fake images, chosen at random across the ENTIRE file list so all
    sub-categories inside "fake" (Stable Diffusion / MidJourney / DALL-E
    are mixed together in this dataset, not split into subfolders) get
    proportional representation. Real and fake are extracted interleaved,
    one of each at a time, so if disk space runs out partway through, what
    we already have is still a balanced partial sample rather than
    real-only or fake-only. The zip is deleted immediately after -- its
    ~52 GB is not needed once the sample is on disk.

    This exists because the straightforward "download and unzip everything"
    approach needs roughly (zip size + extracted size) of free disk at
    once, which does not fit in this notebook's disk budget -- confirmed
    by an actual crash on an earlier run. Skipping the dataset entirely
    instead of sampling from it was tried next and was worse: it removed
    every diffusion-model fake example from training, and the resulting
    model lost the ability to recognise AI-generated images altogether,
    not just fixed the original real-photo false-positive problem.
    """
    target = DATA_DIR / folder_name
    already_extracted = target.exists() and any(
        p for p in target.rglob("*") if p.is_file() and p.suffix.lower() != ".zip"
    )
    if already_extracted:
        print(f"[skip] {slug} already sampled at {target}")
        return

    target.mkdir(parents=True, exist_ok=True)

    existing_zips = list(target.glob("*.zip"))
    if existing_zips:
        zip_path = existing_zips[0]
        print(f"[skip download] zip already present: {zip_path.name}")
    else:
        print(f"\n[download] {slug}  (free disk: {free_gb():.1f} GB)")
        subprocess.run(
            ["kaggle", "datasets", "download", "-d", slug, "-p", str(target)],
            check=True,
        )
        zip_path = next(target.glob("*.zip"))
        print(f"[downloaded] free disk: {free_gb():.1f} GB")

    print(f"[inspect] reading {slug}'s file list (metadata only, nothing extracted yet)")
    real_extracted = fake_extracted = 0
    stopped_early = False

    with zipfile.ZipFile(zip_path) as zf:
        names = [n for n in zf.namelist() if not n.endswith("/")]
        real_members = [n for n in names if any(part.lower() == "real" for part in Path(n).parts)]
        fake_members = [n for n in names if any(part.lower() == "fake" for part in Path(n).parts)]
        print(f"  zip contains {len(real_members):,} real / {len(fake_members):,} fake candidates")

        random.shuffle(real_members)
        random.shuffle(fake_members)
        real_iter = iter(real_members)
        fake_iter = iter(fake_members)

        while real_extracted < per_class_target or fake_extracted < per_class_target:
            if free_gb() < min_free_gb:
                stopped_early = True
                break
            made_progress = False
            if real_extracted < per_class_target:
                name = next(real_iter, None)
                if name is not None:
                    zf.extract(name, target)
                    real_extracted += 1
                    made_progress = True
            if fake_extracted < per_class_target:
                name = next(fake_iter, None)
                if name is not None:
                    zf.extract(name, target)
                    fake_extracted += 1
                    made_progress = True
            if not made_progress:
                break  # both candidate pools exhausted

    zip_path.unlink()
    status = "(stopped early -- low disk headroom)" if stopped_early else "(reached target)"
    print(f"[done] {slug}: extracted {real_extracted:,} real, {fake_extracted:,} fake {status}")
    print(f"  zip deleted, free disk: {free_gb():.1f} GB")


if not USE_LARGE_DATASET_ONLY:
    fetch_partial_by_class(
        "tristanzhang32/ai-generated-images-vs-real-images", "general",
        per_class_target=GENERAL_SAMPLE_PER_CLASS, min_free_gb=MIN_FREE_GB_SAFETY,
    )

fetch("xhlulu/140k-real-and-fake-faces", "faces")

if INCLUDE_CASUAL_PHOTOS:
    # CC0 public domain. ~78.6k images: real selfies (Univ. of Florida CRCV
    # Selfie Dataset) plus everyday candid photos (Flickr30k). Both halves
    # are genuinely casual, unposed, imperfectly-lit real photography --
    # exactly what's missing from the other two "real" sources -- so every
    # image here is used as a REAL example regardless of its Selfie/
    # NonSelfie subfolder (handled explicitly in Step 6 below).
    fetch("jigrubhatt/selfieimagedetectiondataset", "casual")

print(f"\nAll downloads complete. Free disk remaining: {free_gb():.1f} GB")

## Step 6 — Find the class folders automatically

Different datasets name their folders differently (`REAL`/`FAKE`,
`real`/`fake`, `ai_images`, and so on). Rather than hardcode a guess that
silently breaks everything, this scans what was actually extracted and
reports what it found.

**Read this output before continuing.** Both classes must appear.

In [ ]:
IMAGE_EXT = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

REAL_WORDS = ["real", "authentic", "human", "nature"]
FAKE_WORDS = ["fake", "ai", "synthetic", "generated", "artificial"]

def classify_folder(name):
    low = name.lower()
    for w in FAKE_WORDS:
        if low == w or low.startswith(w + "_") or low.endswith("_" + w) or w in low.split("_"):
            return "fake"
    for w in REAL_WORDS:
        if low == w or low.startswith(w + "_") or low.endswith("_" + w) or w in low.split("_"):
            return "real"
    return None

def count_images_direct(folder):
    """Counts images directly inside `folder` (not recursively), so nested
    class folders aren't double-counted by their parents."""
    try:
        return sum(1 for f in os.scandir(folder)
                   if f.is_file() and Path(f.name).suffix.lower() in IMAGE_EXT)
    except OSError:
        return 0

discovered = []
for root, dirs, _ in os.walk(DATA_DIR):
    if INCLUDE_CASUAL_PHOTOS and Path(root).is_relative_to(DATA_DIR / "casual"):
        continue   # handled separately below -- its folder names (Selfie/
                   # NonSelfie) don't match the real/fake keyword scheme
    label = classify_folder(Path(root).name)
    if label is None:
        continue
    n = count_images_direct(root)
    if n > 0:
        discovered.append((root, label, n))

print("Class folders found:\n")
total_real = total_fake = 0
for path, label, n in sorted(discovered):
    print(f"  [{label:4s}] {os.path.relpath(path, DATA_DIR):55s} {n:>7,} images")
    if label == "real":
        total_real += n
    else:
        total_fake += n

print(f"\n  TOTAL real: {total_real:,}")
print(f"  TOTAL fake: {total_fake:,}")
print(f"  TOTAL:      {total_real + total_fake:,}")

assert total_real > 0 and total_fake > 0, (
    "One class has zero images -- folder detection failed. Run the next cell "
    "and share its output."
)

# The casual/selfie dataset has no fake counterpart and its subfolders are
# named Selfie/NonSelfie rather than real/fake, so it's not run through
# classify_folder above -- every image under it is simply a real photo.
casual_real_paths = []
if INCLUDE_CASUAL_PHOTOS:
    casual_dir = DATA_DIR / "casual"
    for root, _, files in os.walk(casual_dir):
        for fname in files:
            if Path(fname).suffix.lower() in IMAGE_EXT:
                casual_real_paths.append(os.path.join(root, fname))
    print(f"\n  Casual/selfie real photos found: {len(casual_real_paths):,}")

In [ ]:
# Diagnostic only — run this if the cell above found nothing or looked wrong.
for root, dirs, fnames in os.walk(DATA_DIR):
    depth = len(Path(root).relative_to(DATA_DIR).parts)
    if depth > 3:
        dirs[:] = []
        continue
    n = sum(1 for f in fnames if Path(f).suffix.lower() in IMAGE_EXT)
    print("  " * depth + f"{Path(root).name}/  ({n} images directly here)")

## Step 7 — Setup and the label-order safeguard

`ImageFolder` assigns class numbers by sorting folder names
alphabetically, which would put `fake` before `real` and silently invert
every prediction the model ever makes. The dataset class below builds its
file list from explicit folder paths instead, so the mapping is fixed by
us, not by alphabetical accident.

In [ ]:
import random
import io
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights
from PIL import Image, ImageFile

ImageFile.LOAD_TRUNCATED_IMAGES = True   # scraped datasets contain partial files

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

CLASS_ORDER = ["real", "fake"]   # index 0 = real, index 1 = fake
INPUT_SIZE = 224
NORMALIZE_MEAN = [0.485, 0.456, 0.406]
NORMALIZE_STD = [0.229, 0.224, 0.225]


class FileListDataset(Dataset):
    def __init__(self, samples, transform=None):
        self.samples = samples
        self.transform = transform

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, index):
        path, label = self.samples[index]
        try:
            image = Image.open(path).convert("RGB")
        except Exception:
            # A few unreadable files is normal at this scale. Substitute
            # neutral grey rather than crashing a 3-hour training run.
            image = Image.new("RGB", (256, 256), (128, 128, 128))
        if self.transform:
            image = self.transform(image)
        return image, label


def build_model(pretrained: bool = False) -> nn.Module:
    """Must stay identical to app/model.py on your laptop."""
    weights = EfficientNet_B0_Weights.DEFAULT if pretrained else None
    model = efficientnet_b0(weights=weights)
    in_features = model.classifier[1].in_features
    model.classifier[1] = nn.Linear(in_features, 1)
    return model


class RandomJPEGRecompression:
    """Randomly re-saves the image through a JPEG encoder at a random
    quality level (or leaves it untouched). Training-set only: without
    this, the model can learn to key on which raw dataset a real/fake
    example happened to be encoded with, rather than genuine content.
    Real-world uploads -- especially anything that's passed through
    WhatsApp or Instagram -- are recompressed far more aggressively than
    any single Kaggle source, so training needs to see that range too.
    """
    def __init__(self, quality_range=(40, 95), p=0.5):
        self.quality_range = quality_range
        self.p = p

    def __call__(self, image):
        if random.random() > self.p:
            return image
        quality = random.randint(*self.quality_range)
        buffer = io.BytesIO()
        image.save(buffer, format="JPEG", quality=quality)
        buffer.seek(0)
        return Image.open(buffer).convert("RGB")


class RandomGaussianNoise:
    """Adds mild sensor-noise-like jitter after ToTensor, for the same
    reason as RandomJPEGRecompression: phone cameras are noisier than
    professionally-shot stock photos or GAN output, and the model
    shouldn't learn to associate "clean" with "real"."""
    def __init__(self, std_range=(0.0, 0.03), p=0.3):
        self.std_range = std_range
        self.p = p

    def __call__(self, tensor):
        if random.random() > self.p:
            return tensor
        std = random.uniform(*self.std_range)
        return tensor + torch.randn_like(tensor) * std


train_transforms = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomCrop(INPUT_SIZE),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.15),
    RandomJPEGRecompression(),
    transforms.ToTensor(),
    RandomGaussianNoise(),
    transforms.Normalize(mean=NORMALIZE_MEAN, std=NORMALIZE_STD),
])

eval_transforms = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.CenterCrop(INPUT_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean=NORMALIZE_MEAN, std=NORMALIZE_STD),
])

print("Setup complete.")

## Step 8 - Build a balanced train / validation / test split

Five things happen here that matter:

**Capping faces.** 70,000 images per class from a single source would
dwarf the disk-limited general-dataset sample and drown it out completely
-- the exact failure mode a previous run hit, just with a different pair
of sources. Capping it keeps the mix genuinely mixed.

**Splitting on UNIQUE files first, oversampling second.** This order
matters and getting it backwards is a subtle but serious bug: if a general
-dataset image is duplicated (for oversampling) BEFORE the train/valid/test
split, its two copies can land on opposite sides of that split by chance --
the model would then effectively train on a file it's also being "tested"
on, quietly inflating the reported test accuracy. So the split happens on
unique files only, and oversampling is applied afterward, strictly inside
the training set. Valid and test stay 100% unique, unduplicated files.

**Oversampling the general sample (training set only).** It's small by
necessity (disk), so each of its images is repeated several times in the
training pool to give it real statistical weight instead of vanishing next
to the much larger faces dataset. Augmentation (Step 7) still randomizes
each repeat independently, so this is not the same as training on literal
duplicate images.

**Casual photos added on top, uncapped.** They have no fake counterpart,
so there's nothing for them to be "paired" against -- they exist purely to
broaden what "real" looks like.

**Splitting by file, with a fixed random seed.** The test set is chosen
once and never seen during training, so the final accuracy is honest.

In [ ]:
random.seed(42)

pool = {"real": {}, "fake": {}}
for folder, label, _ in discovered:
    source = Path(folder).relative_to(DATA_DIR).parts[0]
    bucket = pool[label].setdefault(source, [])
    for entry in os.scandir(folder):
        if entry.is_file() and Path(entry.name).suffix.lower() in IMAGE_EXT:
            bucket.append(entry.path)

for label in pool:
    for source in pool[label]:
        random.shuffle(pool[label][source])
random.shuffle(casual_real_paths)

# Faces is capped so it can't numerically overwhelm the disk-limited
# general sample; casual has no fake counterpart, so it's simply added to
# the real pool. Oversampling general happens LATER, after the split (see
# the markdown above for why doing it before the split would leak files
# across train/valid/test).
faces_real = pool["real"].get("faces", [])[:FACES_CAP_PER_CLASS]
faces_fake = pool["fake"].get("faces", [])[:FACES_CAP_PER_CLASS]
general_real = pool["real"].get("general", [])
general_fake = pool["fake"].get("general", [])
general_path_set = set(general_real) | set(general_fake)

real_unique = faces_real + casual_real_paths + general_real
fake_unique = faces_fake + general_fake
random.shuffle(real_unique)
random.shuffle(fake_unique)

# Balance the two classes on UNIQUE files, and apply the optional cap.
limit = min(len(real_unique), len(fake_unique))
if MAX_PER_CLASS is not None:
    limit = min(limit, MAX_PER_CLASS)

unique_by_label = {"real": real_unique, "fake": fake_unique}
samples = []
for label in CLASS_ORDER:
    for path in unique_by_label[label][:limit]:
        samples.append((path, CLASS_ORDER.index(label)))
random.shuffle(samples)

n_total = len(samples)
n_train = int(n_total * 0.80)
n_valid = int(n_total * 0.10)

train_samples = samples[:n_train]
valid_samples = samples[n_train:n_train + n_valid]
test_samples  = samples[n_train + n_valid:]
n_train_unique, n_valid_final, n_test_final = len(train_samples), len(valid_samples), len(test_samples)

def oversample_general(sample_list, label_value, factor):
    """Duplicates general-dataset entries of one class (factor-1) extra
    times. Only ever called on train_samples -- valid/test must stay
    single-copy so accuracy numbers stay honest (see markdown above)."""
    extra = []
    for path, lab in sample_list:
        if lab == label_value and path in general_path_set:
            extra.extend([(path, lab)] * (factor - 1))
    return sample_list + extra

train_samples = oversample_general(train_samples, CLASS_ORDER.index("real"), GENERAL_REAL_OVERSAMPLE)
train_samples = oversample_general(train_samples, CLASS_ORDER.index("fake"), GENERAL_FAKE_OVERSAMPLE)
random.shuffle(train_samples)

train_dataset = FileListDataset(train_samples, train_transforms)
valid_dataset = FileListDataset(valid_samples, eval_transforms)
test_dataset  = FileListDataset(test_samples,  eval_transforms)

# NOTE: these are CANDIDATE pool sizes going into the class-balancing cut,
# not necessarily the final used count -- if real_unique and fake_unique
# differ in size, the random slice to `limit` is what actually lands in
# the dataset. A previous version of this cell printed these as "used"
# directly, which was misleading for exactly this reason.
print("Real class UNIQUE candidates (before the class-balancing cut):")
print(f"  faces (capped at {FACES_CAP_PER_CLASS:,}): {len(faces_real):>7,}")
print(f"  casual:                        {len(casual_real_paths):>7,}")
print(f"  general (unique, pre-oversample): {len(general_real):>7,}")
print(f"  TOTAL unique candidates:       {len(real_unique):>7,}")

print("Fake class UNIQUE candidates (before the class-balancing cut):")
print(f"  faces (capped at {FACES_CAP_PER_CLASS:,}): {len(faces_fake):>7,}")
print(f"  general (unique, pre-oversample): {len(general_fake):>7,}")
print(f"  TOTAL unique candidates:       {len(fake_unique):>7,}")

print(f"\nUnique files per class after balancing: {limit:,}  (unique total: {n_total:,})")
print(f"  Valid (held out, unique, untouched by oversampling): {n_valid_final:,}")
print(f"  Test  (held out, unique, untouched by oversampling): {n_test_final:,}")
print(f"  Train, unique before oversampling: {n_train_unique:,}")
print(f"  Train, ACTUAL after general-dataset oversampling: {len(train_samples):,}")

def label_balance(sample_list):
    counts = {0: 0, 1: 0}
    for _, lab in sample_list:
        counts[lab] += 1
    return f"real={counts[0]:,} fake={counts[1]:,}"

print(f"\nTrain balance (after oversampling): {label_balance(train_samples)}")
print(f"Test balance (unique, no oversampling): {label_balance(test_samples)}")

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=4, pin_memory=True, drop_last=True)
valid_loader = DataLoader(valid_dataset, batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=4, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=4, pin_memory=True)

## Step 9 — Build the model

The printout below is worth screenshotting for your report: it shows the
original bug in plain numbers.

In [ ]:
model = build_model(pretrained=True).to(device)

def count_trainable(m):
    return sum(p.numel() for p in m.parameters() if p.requires_grad)

def freeze_backbone(m):
    for p in m.features.parameters():
        p.requires_grad = False

def unfreeze_everything(m):
    for p in m.parameters():
        p.requires_grad = True

total = sum(p.numel() for p in model.parameters())
print(f"Total parameters:              {total:,}")

freeze_backbone(model)
head_only = count_trainable(model)
print(f"Phase A trainable (head only): {head_only:,}  ({head_only/total*100:.2f}%)")

for p in model.features[-1].parameters():
    p.requires_grad = True
old_phase_b = count_trainable(model)
print(f"OLD notebook Phase B:          {old_phase_b:,}  ({old_phase_b/total*100:.2f}%)  <-- the bug")

unfreeze_everything(model)
print(f"THIS notebook Phase B:         {count_trainable(model):,}  (100.00%)  <-- the fix")

freeze_backbone(model)
criterion = nn.BCEWithLogitsLoss()

## Step 10 — Training loop

Uses mixed precision, which roughly halves the time per epoch on a T4 with no loss of accuracy. Progress prints every 100 batches so you can tell it's alive during long epochs.

In [ ]:
import time
from torch.amp import autocast, GradScaler

scaler = GradScaler("cuda")

def run_epoch(model, loader, optimizer=None, scheduler=None, tag=""):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()

    total_loss, correct, total = 0.0, 0, 0
    started = time.time()
    context = torch.enable_grad() if is_train else torch.no_grad()

    with context:
        for i, (images, labels) in enumerate(loader):
            images = images.to(device, non_blocking=True)
            labels = labels.float().unsqueeze(1).to(device, non_blocking=True)

            if is_train:
                optimizer.zero_grad(set_to_none=True)

            with autocast("cuda"):
                logits = model(images)
                loss = criterion(logits, labels)

            if is_train:
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
                if scheduler is not None:
                    scheduler.step()

            total_loss += loss.item() * images.size(0)
            predictions = (torch.sigmoid(logits.float()) >= 0.5).float()
            correct += (predictions == labels).sum().item()
            total += images.size(0)

            if is_train and i > 0 and i % 100 == 0:
                elapsed = time.time() - started
                pct = 100 * i / len(loader)
                eta = elapsed / i * (len(loader) - i)
                print(f"    {tag} batch {i}/{len(loader)} ({pct:.0f}%) "
                      f"running_acc={correct/total:.4f} eta={eta/60:.1f}min")

    return total_loss / total, correct / total


CHECKPOINT_PATH = "/content/deepguard_bouncer.pth"

def save_checkpoint(model):
    torch.save(model.state_dict(), CHECKPOINT_PATH)
    if DRIVE_BACKUP_PATH:
        try:
            shutil.copy(CHECKPOINT_PATH, DRIVE_BACKUP_PATH)
        except Exception as e:
            print(f"    (Drive backup failed: {e})")

print("Training functions ready.")

## Step 11 — Phase A: warm up the classifier head

The new head starts random. Training it briefly against a frozen backbone stops the first fine-tuning gradients from being large and destructive.

In [ ]:
freeze_backbone(model)

head_optimizer = torch.optim.Adam(model.classifier.parameters(), lr=1e-3)
best_val_acc = 0.0

for epoch in range(1, EPOCHS_HEAD + 1):
    t0 = time.time()
    train_loss, train_acc = run_epoch(model, train_loader, head_optimizer, tag=f"A{epoch}")
    val_loss, val_acc = run_epoch(model, valid_loader)
    print(f"[Phase A][{epoch}/{EPOCHS_HEAD}] train_acc={train_acc:.4f} | "
          f"val_acc={val_acc:.4f} | {(time.time()-t0)/60:.1f} min")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        save_checkpoint(model)
        print(f"  -> best so far ({val_acc:.4f}), saved.")

## Step 12 — Phase B: fine-tune the whole network

This is the cell that produces the accuracy. Every layer unfreezes, the
learning rate is 1e-4 (ten times the original notebook's), and a cosine
schedule decays it smoothly toward zero so the model settles instead of
oscillating.

**This is the long part.** Each epoch takes roughly 15-35 minutes
depending on the dataset size you chose. Progress prints every 100
batches with an ETA.

In [ ]:
unfreeze_everything(model)

finetune_optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    finetune_optimizer, T_max=EPOCHS_FINETUNE * len(train_loader)
)

patience, stalled = 3, 0

for epoch in range(1, EPOCHS_FINETUNE + 1):
    t0 = time.time()
    train_loss, train_acc = run_epoch(model, train_loader, finetune_optimizer,
                                      scheduler, tag=f"B{epoch}")
    val_loss, val_acc = run_epoch(model, valid_loader)
    lr_now = finetune_optimizer.param_groups[0]["lr"]
    print(f"[Phase B][{epoch}/{EPOCHS_FINETUNE}] train_acc={train_acc:.4f} | "
          f"val_acc={val_acc:.4f} | lr={lr_now:.2e} | {(time.time()-t0)/60:.1f} min")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        stalled = 0
        save_checkpoint(model)
        print(f"  -> best so far ({val_acc:.4f}), saved.")
    else:
        stalled += 1
        print(f"  -> no improvement ({stalled}/{patience})")
        if stalled >= patience:
            print("  -> stopping early.")
            break

print(f"\nBest validation accuracy: {best_val_acc:.4f}")

## Step 13 — Final test on data never seen during training

This is the number you quote to your teacher.

In [ ]:
model.load_state_dict(torch.load(CHECKPOINT_PATH, map_location=device))

test_loss, test_acc = run_epoch(model, test_loader)
print(f"TEST ACCURACY: {test_acc:.4f}   ({test_acc*100:.2f}%)")
print(f"Test set size: {len(test_dataset):,} images never seen during training")
print(f"\nPrevious model: 85.67%, and it only worked on human faces.")

## Step 14 — Detailed breakdown for your report

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report, roc_auc_score

model.eval()
all_probs, all_labels = [], []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        with autocast("cuda"):
            logits = model(images)
        all_probs.extend(torch.sigmoid(logits.float()).cpu().numpy().flatten().tolist())
        all_labels.extend(labels.numpy().tolist())

all_preds = [1 if p >= 0.5 else 0 for p in all_probs]
cm = confusion_matrix(all_labels, all_preds)
tn, fp, fn, tp = cm.ravel()

print("Confusion matrix (rows = actual, cols = predicted), order [real, fake]:")
print(cm)
print(f"\nReal images correctly identified: {tn:,}")
print(f"Real images wrongly called AI:    {fp:,}   (false alarms)")
print(f"AI images wrongly called real:    {fn:,}   (missed fakes)")
print(f"AI images correctly identified:   {tp:,}")
print()
print(classification_report(all_labels, all_preds, target_names=CLASS_ORDER, digits=4))
print(f"ROC-AUC: {roc_auc_score(all_labels, all_probs):.4f}")

## Step 15 — Save and download

In [ ]:
import datetime

sources = ["xhlulu/140k-real-and-fake-faces"]
if not USE_LARGE_DATASET_ONLY:
    sources.append(f"tristanzhang32/ai-generated-images-vs-real-images (disk-safe sample, ~{GENERAL_SAMPLE_PER_CLASS}/class target)")
if INCLUDE_CASUAL_PHOTOS:
    sources.append("jigrubhatt/selfieimagedetectiondataset")

torch.save({
    "model_state_dict": model.state_dict(),
    "class_order": CLASS_ORDER,
    "architecture": "efficientnet_b0",
    "input_size": INPUT_SIZE,
    "normalize_mean": NORMALIZE_MEAN,
    "normalize_std": NORMALIZE_STD,
    "trained_on": sources,
    "scope": "general AI-generated media detection (any subject, not faces-only)",
    "generators_covered": ["Stable Diffusion", "MidJourney", "DALL-E", "StyleGAN"],
    "training_images": n_total,
    "test_accuracy": test_acc,
    "training_version": 5,
    "saved_at_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(),
}, CHECKPOINT_PATH)

print(f"Saved locally to {CHECKPOINT_PATH}. Test accuracy: {test_acc:.4f}  |  Trained on {n_total:,} images")

if DRIVE_BACKUP_PATH:
    # The local save above already succeeded by this point regardless of
    # what happens next -- Drive's mounted filesystem has been flaky all
    # session, and an uncaught error here previously looked like the whole
    # save had failed when it hadn't. This is a nice-to-have backup, not
    # the actual deliverable, so it must not be able to halt the notebook.
    try:
        shutil.copy(CHECKPOINT_PATH, DRIVE_BACKUP_PATH)
        print(f"Also backed up to Drive: {DRIVE_BACKUP_PATH}")
    except Exception as e:
        print(f"(Drive backup failed, but the local file above is fine: {e})")

In [ ]:
from google.colab import files
files.download(CHECKPOINT_PATH)
print("Replace models/deepguard_bouncer.pth on your laptop with this file.")